# Bayesian Statistics and MAP Estimation

## Learning Objectives
- Contrast the **frequentist** view ($\theta$ is a fixed unknown constant) with the **Bayesian** view ($\theta$ is a random variable)
- Derive the **posterior distribution** $p(\theta|S)$ via Bayes' rule
- Define the **MAP (Maximum A Posteriori)** estimate as a point approximation to the full posterior
- Show that a Gaussian prior $\theta \sim \mathcal{N}(0, \tau^2 I)$ on logistic regression recovers **L2 regularisation**
- Visualise how the prior shrinks parameters toward zero to reduce overfitting

## Problem Statement

**Frequentist ML (MLE):**
$$\theta_{\text{ML}} = \arg\max_\theta \prod_{i=1}^m p(y^{(i)} | x^{(i)}; \theta)$$

**Bayesian approach:** treat $\theta$ as a random variable with prior $p(\theta)$. After observing data $S$:
$$p(\theta|S) = \frac{\left(\prod_{i=1}^m p(y^{(i)}|x^{(i)}, \theta)\right) p(\theta)}{p(S)}$$

**Fully Bayesian prediction** integrates over all $\theta$:
$$p(y|x, S) = \int_\theta p(y|x, \theta)\, p(\theta|S)\, d\theta$$

This integral is usually intractable, so in practice we use the **MAP estimate**:
$$\theta_{\text{MAP}} = \arg\max_\theta \prod_{i=1}^m p(y^{(i)}|x^{(i)}, \theta)\, p(\theta)$$

MAP = MLE + prior term. With Gaussian prior $\theta \sim \mathcal{N}(0, \tau^2 I)$, MAP becomes L2-regularised MLE.

## 1. Frequentist vs. Bayesian View

**Frequentist:** $\theta$ is a fixed, unknown constant. Repeated experiments give different data but the same $\theta$.
**Bayesian:** $\theta$ is a random variable. Before seeing data we have a prior; after seeing data we update to the posterior.

The key equation:
$$\underbrace{p(\theta|S)}_{\text{posterior}} \propto \underbrace{p(S|\theta)}_{\text{likelihood}} \cdot \underbrace{p(\theta)}_{\text{prior}}$$

Below we visualise this update for a simple Bernoulli coin-flip problem (Beta-Binomial conjugate pair).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist

np.random.seed(0)

# True coin bias
theta_true = 0.3

# Beta(2, 2) prior — mild preference for balanced coins
prior_a, prior_b = 2, 2

theta_grid = np.linspace(0, 1, 500)
observations = [0, 1, 5, 20, 100]

fig, axes = plt.subplots(1, len(observations), figsize=(16, 4.5))

heads_seen = 0
flips_seen = 0

for ax, n_new in zip(axes, observations):
    if n_new > flips_seen:
        new_flips = np.random.binomial(1, theta_true, n_new - flips_seen)
        heads_seen += new_flips.sum()
        flips_seen  = n_new

    # Posterior: Beta(a + heads, b + tails)
    a_post = prior_a + heads_seen
    b_post = prior_b + (flips_seen - heads_seen)

    prior_pdf     = beta_dist.pdf(theta_grid, prior_a, prior_b)
    posterior_pdf = beta_dist.pdf(theta_grid, a_post,  b_post)
    likelihood    = theta_grid ** heads_seen * (1 - theta_grid) ** (flips_seen - heads_seen)
    likelihood   /= (likelihood.max() + 1e-12)

    ax.plot(theta_grid, prior_pdf,     'g--', lw=1.5, label='Prior $p(\\theta)$')
    ax.plot(theta_grid, likelihood,    'b:',  lw=1.5, label='Likelihood (normalised)')
    ax.plot(theta_grid, posterior_pdf, 'r-',  lw=2.5, label='Posterior $p(\\theta|S)$')
    ax.axvline(theta_true, color='k', ls='--', lw=1.5, label=f'True $\\theta={theta_true}$')

    map_est = (a_post - 1) / (a_post + b_post - 2)
    mle_est = heads_seen / flips_seen if flips_seen > 0 else 0.5

    ax.set_title(f'$m={flips_seen}$ flips, {heads_seen} heads\n'
                 f'MLE={mle_est:.2f}, MAP={map_est:.2f}', fontsize=8.5)
    ax.set_xlabel('$\\theta$')
    if ax == axes[0]:
        ax.set_ylabel('Density'); ax.legend(fontsize=7)

fig.suptitle('Bayesian Update: Prior → Posterior as Data Accumulates\n'
             'Posterior concentrates on true $\\theta$ as $m$ grows',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. MAP Estimate Derivation

The MAP estimate approximates the full posterior by finding its mode:
$$\theta_{\text{MAP}} = \arg\max_\theta \log p(\theta|S) = \arg\max_\theta \left[\sum_{i=1}^m \log p(y^{(i)}|x^{(i)}, \theta) + \log p(\theta)\right]$$

**Gaussian prior → L2 regularisation:**
If $p(\theta) = \mathcal{N}(0, \tau^2 I)$ then:
$$\log p(\theta) = -\frac{1}{2\tau^2}\|\theta\|^2 + \text{const}$$

So MAP for logistic regression becomes:
$$\theta_{\text{MAP}} = \arg\max_\theta \sum_{i=1}^m \log p(y^{(i)}|x^{(i)}, \theta) - \frac{\lambda}{2}\|\theta\|^2, \qquad \lambda = \frac{1}{\tau^2}$$

This is exactly **L2-regularised logistic regression** — the Bayesian view provides a principled derivation of regularisation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Demonstrate MAP = MLE + prior for 1D linear regression
# True model: y = 0.5x + noise; prior: theta ~ N(0, tau^2)

np.random.seed(7)

theta_true = 0.5
m          = 10
X_1d       = np.random.uniform(-2, 2, m)
y_1d       = theta_true * X_1d + np.random.normal(0, 1.5, m)

theta_grid = np.linspace(-2, 3, 500)

def log_likelihood(theta, X, y, sigma=1.5):
    resid = y - theta * X
    return -np.sum(resid**2) / (2 * sigma**2)

def log_prior(theta, tau):
    return -theta**2 / (2 * tau**2)

taus = [0.3, 1.0, 5.0]   # small tau = strong prior (heavy regularisation)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: log-posterior as a function of theta for different tau
ax = axes[0]
ll = np.array([log_likelihood(t, X_1d, y_1d) for t in theta_grid])
ax.plot(theta_grid, ll - ll.max(), 'k-', lw=3, label='Log-likelihood (MLE)')

colors = ['#2166ac', '#d6604d', '#4dac26']
for tau, color in zip(taus, colors):
    lp      = np.array([log_prior(t, tau) for t in theta_grid])
    lpost   = ll + lp
    map_est = theta_grid[np.argmax(lpost)]
    ax.plot(theta_grid, lpost - lpost.max(), '-', color=color, lw=2,
            label=f'Log-posterior $\\tau={tau}$, MAP={map_est:.2f}')

mle = theta_grid[np.argmax(ll)]
ax.axvline(theta_true, color='gray',  ls=':',  lw=1.5, label=f'True $\\theta={theta_true}$')
ax.axvline(mle,        color='black', ls='--', lw=1.5, label=f'MLE={mle:.2f}')
ax.set_xlabel('$\\theta$'); ax.set_ylabel('Normalised log-posterior')
ax.set_title('MAP shifts MLE toward zero as prior gets stronger\n'
             '(smaller $\\tau$ = stronger L2 regularisation)')
ax.legend(fontsize=8.5); ax.set_ylim(-4, 0.5)

# Right: MAP estimate vs. MLE as a function of tau (= 1/sqrt(lambda))
ax = axes[1]
tau_range = np.logspace(-1, 1.5, 100)
map_ests  = []
for tau in tau_range:
    lp    = np.array([log_prior(t, tau) for t in theta_grid])
    lpost = ll + lp
    map_ests.append(theta_grid[np.argmax(lpost)])

ax.semilogx(tau_range, map_ests, 'b-', lw=2.5, label='MAP estimate')
ax.axhline(mle,        color='black', ls='--', lw=2, label=f'MLE = {mle:.2f}')
ax.axhline(theta_true, color='gray',  ls=':',  lw=1.5, label=f'True $\\theta = {theta_true}$')
ax.axhline(0,          color='red',   ls=':',  lw=1,   label='Prior mean = 0')
ax.set_xlabel('Prior width $\\tau$ (larger = weaker prior / less regularisation)')
ax.set_ylabel('Parameter estimate')
ax.set_title('MAP interpolates between prior mean (0) and MLE\n'
             '$\\tau \\to 0$: MAP → 0;  $\\tau \\to \\infty$: MAP → MLE')
ax.legend(fontsize=9)

fig.suptitle('MAP Estimation: Gaussian Prior $\\theta \\sim \\mathcal{N}(0, \\tau^2)$ → L2 Regularisation',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. MAP vs. MLE: Overfitting Reduction

With Gaussian prior $\theta \sim \mathcal{N}(0, \tau^2 I)$:

$$\theta_{\text{MAP}} = \arg\max_\theta \sum_i \log p(y^{(i)}|x^{(i)}, \theta) - \frac{1}{2\tau^2}\|\theta\|^2$$

The prior penalises large $\|\theta\|$ — parameters are pulled toward zero, reducing the effective complexity of the model.

In the `sklearn` API this is `LogisticRegression(C=tau^2)` — `C` is the inverse regularisation strength.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

np.random.seed(0)

# High-dimensional problem: n >> m to make overfitting visible
X, y = make_classification(
    n_samples=80, n_features=50, n_informative=5,
    n_redundant=0, random_state=0
)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

# C = tau^2: large C = weak prior (close to MLE), small C = strong prior (heavy regularisation)
C_values   = np.logspace(-3, 3, 40)
train_accs, test_accs, theta_norms = [], [], []

for C in C_values:
    clf = LogisticRegression(C=C, max_iter=1000, random_state=0)
    clf.fit(X_tr, y_tr)
    train_accs.append(clf.score(X_tr, y_tr))
    test_accs.append(clf.score(X_te, y_te))
    theta_norms.append(np.linalg.norm(clf.coef_))

best_C = C_values[np.argmax(test_accs)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: train/test accuracy vs. C (= tau^2)
ax = axes[0]
ax.semilogx(C_values, train_accs, 'b-', lw=2.5, label='Train accuracy (MLE gets this high)')
ax.semilogx(C_values, test_accs,  'r-', lw=2.5, label='Test accuracy (what we care about)')
ax.axvline(best_C, color='green', ls='--', lw=2,
           label=f'Best $C = \\tau^2 = {best_C:.3f}$\n($\\lambda = 1/C = {1/best_C:.1f}$)')
ax.axvline(C_values[-1], color='gray', ls=':', lw=1.5, alpha=0.7)
ax.text(C_values[-1]*0.6, 0.52, 'MLE\n(no prior)', fontsize=8, color='gray', ha='right')
ax.text(C_values[0]*1.8,  0.52, 'Strong\nprior', fontsize=8, color='gray')
ax.set_xlabel('$C = \\tau^2$ (sklearn: inverse regularisation strength)')
ax.set_ylabel('Accuracy')
ax.set_title('MAP (L2 reg.) reduces overfitting\n'
             '($n=50$ features, $m=56$ train examples)')
ax.legend(fontsize=8.5); ax.set_ylim(0.45, 1.05)

# Right: theta norm vs. C
ax = axes[1]
ax.semilogx(C_values, theta_norms, 'purple', lw=2.5)
ax.axvline(best_C, color='green', ls='--', lw=2, label=f'Best $C={best_C:.3f}$')
ax.set_xlabel('$C = \\tau^2$')
ax.set_ylabel('$\\|\\theta_{\\text{MAP}}\\|$')
ax.set_title('Gaussian prior shrinks $\\|\\theta\\|$ toward zero\n'
             'Small $C$ (strong prior) → small norm → less overfitting')
ax.legend(fontsize=9)

fig.suptitle('Gaussian Prior = L2 Regularisation: Bayesian Interpretation of Regularisation',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Best C (tau^2): {best_C:.4f}  →  lambda = 1/C = {1/best_C:.2f}')

## 4. Frequentist vs. Bayesian vs. MAP

| Approach | Parameter | Estimate | Regularisation |
|---|---|---|---|
| Frequentist / MLE | Fixed constant | $\arg\max_\theta \prod_i p(y^{(i)}|x^{(i)};\theta)$ | None |
| Full Bayesian | Random variable | $p(\theta|S)$ (full posterior) | Implicit via prior |
| MAP | Random variable | Mode of $p(\theta|S)$ | Explicit: prior term |
| L2-reg. MLE | Fixed constant | $\arg\max_\theta \mathcal{L}(\theta) - \frac{\lambda}{2}\|\theta\|^2$ | $\equiv$ MAP with $p(\theta)=\mathcal{N}(0,\frac{1}{\lambda}I)$ |
| L1-reg. MLE | Fixed constant | $\arg\max_\theta \mathcal{L}(\theta) - \lambda\|\theta\|_1$ | $\equiv$ MAP with Laplace prior |

**Key insight:** MAP with a Gaussian prior is equivalent to L2 regularisation. The prior width $\tau$ controls the regularisation strength: $\lambda = 1/\tau^2$. The Bayesian framework thus provides a principled, interpretable derivation of regularisation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, laplace

# Visualise Gaussian prior (L2) vs Laplace prior (L1) — different shapes explain
# why L1 promotes sparsity (the Laplace prior has a sharp peak at 0)

theta = np.linspace(-4, 4, 500)
tau   = 1.0

gaussian_prior = norm.pdf(theta, 0, tau)
laplace_prior  = laplace.pdf(theta, 0, tau / np.sqrt(2))  # same variance

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: prior shapes
ax = axes[0]
ax.plot(theta, gaussian_prior, 'b-', lw=2.5,
        label='Gaussian $\\mathcal{N}(0,\\tau^2)$ → L2 reg')
ax.plot(theta, laplace_prior,  'r-', lw=2.5,
        label='Laplace → L1 reg (promotes sparsity)')
ax.axvline(0, color='gray', ls='--', lw=1)
ax.set_xlabel('$\\theta$'); ax.set_ylabel('Prior density $p(\\theta)$')
ax.set_title('Prior shapes and implied regularisation\nLaplace has heavier tails + sharper peak at 0')
ax.legend(fontsize=9)
ax.text(0.55, 0.92,
        'Gaussian: penalises $\\|\\theta\\|^2$\n→ L2 shrinkage (never exactly 0)',
        transform=ax.transAxes, fontsize=8.5,
        bbox=dict(boxstyle='round', fc='#cce5ff', alpha=0.9))
ax.text(0.03, 0.60,
        'Laplace: penalises $|\\theta|$\n→ L1 sparsity (some params = 0)',
        transform=ax.transAxes, fontsize=8.5,
        bbox=dict(boxstyle='round', fc='#f8d7da', alpha=0.9))

# Right: -log prior = regularisation penalty
ax = axes[1]
ax.plot(theta, -np.log(gaussian_prior + 1e-12), 'b-', lw=2.5,
        label=r'$-\log p(\theta)$ Gaussian $\propto \|\theta\|^2$ (L2)')
ax.plot(theta, -np.log(laplace_prior  + 1e-12), 'r-', lw=2.5,
        label=r'$-\log p(\theta)$ Laplace $\propto |\theta|$ (L1)')
ax.set_xlabel('$\\theta$')
ax.set_ylabel('Regularisation penalty $-\\log p(\\theta)$')
ax.set_title('MAP optimisation adds $-\\log p(\\theta)$ to the negative log-likelihood\n'
             'This IS the regularisation term')
ax.legend(fontsize=9); ax.set_ylim(0, 12)

fig.suptitle('Prior Shape = Regularisation Penalty\n'
             'Gaussian prior ↔ L2 reg   |   Laplace prior ↔ L1 reg (LASSO)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="464"
     viewBox="0 0 780 464" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>

  <!-- Row 1: Prior -->
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Prior p(&#x3b8;)</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">+ training data S</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >Bayesian view: &#x3b8; is a random variable, not a fixed constant</text>

  <!-- step 1&#x2192;2 -->
  <line x1="102" y1="58" x2="102" y2="108"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="82" font-size="11.5" font-style="italic" fill="#555"
        >Bayes&#x2019; rule: p(&#x3b8;|S) &#x221d; p(S|&#x3b8;) p(&#x3b8;)</text>

  <!-- Row 2: Posterior -->
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="140" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Posterior p(&#x3b8;|S)</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >full distribution over &#x3b8; &#x2014; intractable for most models</text>

  <!-- step 2&#x2192;3 -->
  <line x1="102" y1="158" x2="102" y2="208"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="178" font-size="11.5" font-style="italic" fill="#555"
        >approximate by taking mode of posterior</text>

  <!-- Row 3: MAP -->
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="240" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">MAP estimate</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >argmax &#x2211; log p(y&#x2071;|x&#x2071;,&#x3b8;) + log p(&#x3b8;) = MLE + prior term</text>

  <!-- step 3&#x2192;4 -->
  <line x1="102" y1="258" x2="102" y2="308"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="282" font-size="11.5" font-style="italic" fill="#555"
        >substitute Gaussian prior: log p(&#x3b8;) = &#x2212;&#x2016;&#x3b8;&#x2016;&#xb2;/(2&#x3c4;&#xb2;)</text>

  <!-- Row 4: L2 reg -->
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="340" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">L2 regularisation</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >argmax &#x2211; log p(y&#x2071;|x&#x2071;,&#x3b8;) &#x2212; (&#x3bb;/2)&#x2016;&#x3b8;&#x2016;&#xb2;, where &#x3bb; = 1/&#x3c4;&#xb2;</text>

  <!-- step 4&#x2192;5 -->
  <line x1="102" y1="358" x2="102" y2="408"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>

  <!-- Row 5: Regularised model -->
  <rect x="10" y="412" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="440" font-size="13.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Regularised model</text>
  <line x1="197" y1="435" x2="216" y2="435"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="412" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="440" font-size="13" text-anchor="middle" fill="#333"
        >prior width &#x3c4; chosen by cross-validation &#x2014; Laplace prior gives L1 / LASSO</text>
</svg>
""")

## Summary

| Concept | Formula / Statement | Key Insight |
|---|---|---|
| MLE | $\arg\max_\theta \prod_i p(y^{(i)}\lvert x^{(i)};\theta)$ | Frequentist; no prior; can overfit |
| Posterior | $p(\theta\lvert S) \propto p(S\lvert\theta)\,p(\theta)$ | Bayesian; full distribution; usually intractable |
| MAP | $\arg\max_\theta \sum_i \log p(y^{(i)}\lvert x^{(i)},\theta) + \log p(\theta)$ | Mode of posterior; tractable point estimate |
| Gaussian prior | $p(\theta) = \mathcal{N}(0,\tau^2 I)$ | MAP $\equiv$ L2-regularised MLE with $\lambda = 1/\tau^2$ |
| Laplace prior | $p(\theta) \propto e^{-\lvert\theta\rvert/b}$ | MAP $\equiv$ L1-regularised MLE (LASSO) — promotes sparsity |
| Prior width $\tau$ | Controls regularisation strength | Small $\tau$: strong prior, heavy shrinkage; large $\tau$: weak prior, approaches MLE |

**Key insight:** regularisation has a Bayesian interpretation — the penalty term in L2/L1 regression is exactly $-\log p(\theta)$ for a Gaussian/Laplace prior. MAP provides a principled, probabilistic derivation of what practitioners have long applied as an ad-hoc trick.